In [2]:
import numpy as np
import pandas as pd
path = './cp/lhs_dataset_40.csv'
df = pd.read_csv(path)

conlumns = df.columns.tolist() 
x = conlumns[:23]
y = ['M_ult', 'N_ult']
np.savetxt('cp/x_columns.txt', x, fmt='%s')
np.savetxt('cp/y_columns.txt', y, fmt='%s')

In [3]:
import numpy as np
import pandas as pd
path = './FEA/results_40.csv'
df = pd.read_csv(path)

conlumns = df.columns.tolist() 
x = conlumns[1:20]
y = conlumns[20:]
np.savetxt('FEA/x_columns.txt', x, fmt='%s')
np.savetxt('FEA/y_columns.txt', y, fmt='%s')

In [4]:
import numpy as np
import pandas as pd
path = './me/XY_600_SIM-PAN-50_AutoSklearn_40.csv'
df = pd.read_csv(path)

conlumns = df.columns.tolist() 
x = conlumns[:-3]
y = conlumns[-3:]
np.savetxt('me/pan_x_columns.txt', x, fmt='%s')
np.savetxt('me/pan_y_columns.txt', y, fmt='%s')

In [5]:
import numpy as np
import pandas as pd
path = './me/XY_600_SIM-TRC_AutoSklearn_40.csv'
df = pd.read_csv(path)

conlumns = df.columns.tolist() 
x = conlumns[:-1]
y = conlumns[-1:]
np.savetxt('me/trc_x_columns.txt', x, fmt='%s')
np.savetxt('me/trc_y_columns.txt', y, fmt='%s')

In [9]:
def data_chooice(data_idx):
    if data_idx == 'cp':
        with open('cp/x_columns.txt', 'r') as f:
            x_col = f.readlines()
        with open('cp/y_columns.txt', 'r') as f:
            y_col = f.readlines()
        return x_col, y_col
    elif data_idx == 'FEA':
        with open('FEA/x_columns.txt', 'r') as f:
            x_col = f.readlines()
        with open('FEA/y_columns.txt', 'r') as f:
            y_col = f.readlines()
        return x_col, y_col
    elif data_idx == 'me_pan':
        with open('me/pan_x_columns.txt', 'r') as f:
            x_col = f.readlines()
        with open('me/pan_y_columns.txt', 'r') as f:
            y_col = f.readlines()
        return x_col, y_col
    elif data_idx == 'me_trc':
        with open('me/trc_x_columns.txt', 'r') as f:
            x_col = f.readlines()
        with open('me/trc_y_columns.txt', 'r') as f:
            y_col = f.readlines()
        return x_col, y_col


In [11]:
x,y = data_chooice('me_trc')

In [16]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

DATASETS = {
    "cp": ("cp/x_columns.txt", "cp/y_columns.txt"),
    "FEA": ("FEA/x_columns.txt", "FEA/y_columns.txt"),
    "me_pan": ("me/pan_x_columns.txt", "me/pan_y_columns.txt"),
    "me_trc": ("me/trc_x_columns.txt", "me/trc_y_columns.txt"),
}


def data_choice(data_idx):

    if data_idx not in DATASETS:
        raise ValueError(f"Unknown dataset key: {data_idx}")
    x_path, y_path = DATASETS[data_idx]
    with open(x_path, "r") as f:
        x_col = [line.strip() for line in f if line.strip()]
    with open(y_path, "r") as f:
        y_col = [line.strip() for line in f if line.strip()]
    return x_col, y_col


def data_loader(csv_path, func_name, dataset_key):
    if dataset_key == "me":
        if "pan" in func_name.lower():
            x_cols, y_cols = data_choice("me_pan")
        elif "trc" in func_name.lower():
            x_cols, y_cols = data_choice("me_trc")
        else:
            raise ValueError("For dataset_key='me', func_name must contain 'pan' or 'trc'.")
    else:
        x_cols, y_cols = data_choice(dataset_key)

    df = pd.read_csv(csv_path)
    missing = [c for c in x_cols + y_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")

    X = df[x_cols].to_numpy()
    y = df[y_cols].to_numpy()
    return X, y


def data_path(folder_path):
    folder = Path(folder_path)
    if not folder.exists() or not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
    dataset_key = folder.name
    csv_files = sorted(folder.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {folder}")

    for csv_path in csv_files:
        func_name = csv_path.stem
        print(func_name, csv_path)
        yield data_loader(csv_path, func_name, dataset_key)

x, y = data_path('./me')

XY_600_SIM-PAN-1000_AutoSklearn_40 me\XY_600_SIM-PAN-1000_AutoSklearn_40.csv


ValueError: Missing columns in CSV: ['Mass fraction of SiO2 in solute']